##**Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Bronze
### Objetivo
Ingerir os dados brutos de itens de pedido na camada Bronze do Delta Lake, sem transformações no conteúdo — apenas adicionando colunas de auditoria e rastreabilidade.
### Origem e Destino
| Item | Valor |
| **Origem** | `real-time-data/<snapshot>/ecommerce_itens_pedido.parquet` (ADLS) |
| **Destino** | `squad2/bronze/ecommerce_itens_pedido` (Delta Lake) |
| **Checkpoint** | `squad2/control/ecommerce_itens_pedido/control_file.json` |
| **Modo de escrita** | `append` incremental por snapshot |
### Colunas de Auditoria Adicionadas
| Coluna | Descrição |
| `_snapshot_id` | ID do snapshot de origem (`YYYY/MM/DD/HHMMSS`) |
| `_ingested_at` | Timestamp de ingestão na Bronze |
| `_source` | Fonte dos dados (`real-time-data`) |
| `_camada` | Camada atual (`bronze`) |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | Conexão ADLS, leitura Parquet, escrita Delta, checkpoint e log |

In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA      = "ecommerce_itens_pedido"
BRONZE_PATH = get_delta_path("bronze", TABELA)

inicio = log_inicio(f"feat_squad2_bronze_{TABELA}")

log.info(f"Tabela      : {TABELA}")
log.info(f"Bronze Path : {BRONZE_PATH}")


In [0]:
try:
    snapshots = sorted(listar_snapshots())
    log.info(f"{len(snapshots)} snapshot(s) disponível(is):\n")
    for snap in snapshots:
        print(f"  Pacote {snap}")

except Exception as e:
    log.error(f"Erro ao listar snapshots: {str(e)}")
    raise

- Verificação do Checkpoint
Lê o arquivo de controle JSON para identificar quais snapshots já foram processados.
Apenas snapshots **novos** serão ingeridos nesta execução.

In [0]:
try:
    processados = ler_checkpoint("bronze", TABELA)
    novos       = [s for s in snapshots if s not in processados]
    log.info(f"{len(novos)} snapshot(s) novo(s) para processar")

except Exception as e:
    log.error(f"Erro ao verificar checkpoint: {str(e)}")
    raise

- Ingestão na Camada Bronze (Delta Lake)
Para cada snapshot novo:
1. Lê o arquivo `.parquet` do ADLS
2. Adiciona as colunas de auditoria
3. Grava na tabela Delta em modo `append`
4. Atualiza o checkpoint ao final

In [0]:
try:
    total_linhas = 0

    if not novos:
        log.info("Nenhum snapshot novo para processar!")
    else:
        for snapshot_id in novos:
            log.info(f"Processando: {snapshot_id}")

            df        = ler_parquet(snapshot_id, TABELA)
            df_bronze = df \
                .withColumn("_snapshot_id", lit(snapshot_id)) \
                .withColumn("_ingested_at", current_timestamp()) \
                .withColumn("_source",      lit("real-time-data")) \
                .withColumn("_camada",      lit("bronze"))

            sucesso       = gravar_delta(df_bronze, "bronze", TABELA)
            count         = df_bronze.count()
            total_linhas += count
            processados.add(snapshot_id)

            log.info(f"  OK {snapshot_id} → {count} linhas")

        salvar_checkpoint("bronze", TABELA, processados)
        log.info(f" Total gravado: {total_linhas} linhas")

except Exception as e:
    log.error(f"Erro na ingestão Bronze: {str(e)}")
    raise


- Validação da Camada Bronze
Confirma a gravação lendo a tabela Delta e exibindo contagem, schema e amostra.

In [0]:
try:
    df_bronze = ler_delta("bronze", TABELA)
    total     = df_bronze.count()

    log.info(f" Validação Bronze OK!")
    log.info(f"   Path            : {BRONZE_PATH}")
    log.info(f"   Total registros : {total}")
    log.info(f"   Colunas         : {len(df_bronze.columns)}")

    print("\n Schema Bronze:")
    df_bronze.printSchema()

    print("\n Amostra:")
    display(df_bronze)

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

- Histórico Delta
Exibe o histórico de versões da tabela Delta com retry automático (até 3 tentativas).
Falha no histórico não indica problema nos dados gravados.

In [0]:
try:
    from deltalake import DeltaTable
    import time

    for tentativa in range(3):
        try:
            dt        = DeltaTable(BRONZE_PATH, storage_options=get_storage_options())
            historico = dt.history()
            log.info(f" Histórico Delta: {len(historico)} versão(ões)")
            for h in historico:
                print(f"  v{h['version']} | {h['timestamp']} | {h['operation']}")
            break
        except Exception as e:
            if tentativa < 2:
                log.warning(f"Tentativa {tentativa + 1} falhou — aguardando 5s...")
                time.sleep(5)
            else:
                log.warning(" Histórico indisponível no momento.")

except Exception as e:
    log.warning(f"⚠️ Histórico ignorado: {str(e)[:80]}")

In [0]:
log_fim(f"feat_squad2_bronze_{TABELA}", inicio)